# Tutorial 2: Working with EXFOR (Experimental Nuclear Reaction Data)

## Overview

EXFOR (Exchange Format) is the international database of experimental nuclear reaction data maintained by the IAEA. Unlike ENDF (which contains evaluated data), EXFOR contains raw experimental measurements from publications.

### What you'll learn:
- How to access EXFOR data through the IAEA API
- How to search for specific reactions and isotopes
- How to extract experimental cross-section measurements
- How to visualize experimental data with uncertainties
- How to compare experimental data with evaluated data

### Prerequisites:
```bash
pip install requests matplotlib numpy pandas
```

## 1. Understanding EXFOR

### Key Concepts:

- **Experimental Data**: Direct measurements from experiments, not evaluations
- **Uncertainties**: Experimental data includes measurement uncertainties
- **Entry Numbers**: Each experiment has a unique EXFOR entry number
- **Subentry**: Individual measurements within an entry
- **REACTION**: Specifies what was measured (e.g., U-235(n,f) = neutron-induced fission of U-235)

### Why EXFOR is important:
- Source data for evaluations
- Validation of evaluated libraries
- Uncertainty quantification
- Machine learning training data

In [ ]:
import requests
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path

# Create data directory
data_dir = Path('../data/exfor')
data_dir.mkdir(parents=True, exist_ok=True)

print("Setup complete!")

## 2. Accessing EXFOR via IAEA API

The IAEA provides a REST API for accessing EXFOR data. We'll use the `requests` library to interact with it.

In [ ]:
# IAEA EXFOR API base URL
BASE_URL = "https://nds.iaea.org/exfor/servlet/X4sGetData"

def search_exfor(target, reaction_type, projectile='n'):
    """
    Search EXFOR database for specific reaction data.
    
    Parameters:
    -----------
    target : str
        Target nucleus (e.g., 'U-235', 'Fe-56')
    reaction_type : str
        Reaction type (e.g., 'F' for fission, 'G' for capture, 'EL' for elastic)
    projectile : str
        Projectile particle (default: 'n' for neutron)
    
    Returns:
    --------
    dict : API response containing experimental data
    """
    params = {
        'target': target,
        'reaction': reaction_type,
        'inc': projectile,
        'format': 'json'
    }
    
    try:
        response = requests.get(BASE_URL, params=params, timeout=30)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"Error accessing EXFOR API: {e}")
        return None

print("EXFOR search function defined")

## 3. Alternative: Direct CSV Download

IAEA also provides a web interface where you can download data in various formats. Let's create a helper function to parse EXFOR data.

In [ ]:
def fetch_exfor_cross_section(target, reaction='F', energy_min=1e-2, energy_max=1e7):
    """
    Fetch cross-section data from EXFOR using the IAEA web service.
    
    Note: This is a simplified version. The actual EXFOR API requires
    more specific parameters. For production use, consider using the
    x4i3 Python package or the JANIS interface.
    """
    # For this tutorial, we'll use a direct approach with IAEA's retrieval system
    # URL format for EXFOR data retrieval
    base_url = "https://www-nds.iaea.org/exfor/servlet/X4sGetTabServ"
    
    # Parse target to get Z and A
    # Example: 'U-235' -> Z=92, A=235
    element_map = {'U': 92, 'Pu': 94, 'Fe': 26, 'H': 1, 'C': 6, 'O': 8}
    
    if '-' in target:
        elem, mass = target.split('-')
        z = element_map.get(elem, 92)
        a = int(mass)
    else:
        z, a = 92, 235  # Default to U-235
    
    # Construct query parameters
    params = {
        'targZ': str(z).zfill(3),
        'targA': str(a).zfill(3),
        'Proj': 'n',
        'Reac': reaction
    }
    
    print(f"Searching for: {target}(n,{reaction}) reaction")
    print(f"Parameters: Z={z}, A={a}")
    print("\nNote: EXFOR API requires specific formatting.")
    print("For this tutorial, we'll use simulated EXFOR-like data.")
    print("For real applications, use the JANIS interface or x4i3 package.")
    
    return None

# Try to fetch U-235 fission data
result = fetch_exfor_cross_section('U-235', 'F')

## 4. Working with Actual EXFOR Data

Since the EXFOR API can be complex, let's demonstrate with a realistic example using simulated experimental data that resembles EXFOR format.

In [ ]:
def create_exfor_sample_data():
    """
    Create sample data that resembles EXFOR experimental measurements.
    In practice, you would download this from IAEA.
    """
    # Simulate 3 different experiments measuring U-235 fission cross-section
    np.random.seed(42)
    
    experiments = []
    
    # Experiment 1: Thermal to epithermal range
    exp1_energy = np.array([0.0253, 0.1, 0.5, 1.0, 5.0, 10.0])  # eV
    exp1_xs = np.array([584.4, 280.2, 125.3, 100.5, 55.2, 40.1])  # barns
    exp1_err = exp1_xs * 0.05  # 5% uncertainty
    
    experiments.append({
        'entry': '12345001',
        'author': 'Smith et al.',
        'year': 2018,
        'energy': exp1_energy,
        'xs': exp1_xs,
        'dxs': exp1_err,
        'facility': 'LANSCE'
    })
    
    # Experiment 2: Resonance region
    exp2_energy = np.logspace(1, 3, 20)  # 10 eV to 1 keV
    exp2_xs = 50 + 30 * np.sin(np.log(exp2_energy)) + np.random.normal(0, 3, len(exp2_energy))
    exp2_err = exp2_xs * 0.08  # 8% uncertainty
    
    experiments.append({
        'entry': '12345002',
        'author': 'Jones et al.',
        'year': 2020,
        'energy': exp2_energy,
        'xs': exp2_xs,
        'dxs': exp2_err,
        'facility': 'n_TOF CERN'
    })
    
    # Experiment 3: Fast neutron range
    exp3_energy = np.logspace(5, 7, 15)  # 100 keV to 10 MeV
    exp3_xs = 2.0 + 0.5 * np.log10(exp3_energy/1e5) + np.random.normal(0, 0.1, len(exp3_energy))
    exp3_err = exp3_xs * 0.06  # 6% uncertainty
    
    experiments.append({
        'entry': '12345003',
        'author': 'Brown et al.',
        'year': 2019,
        'energy': exp3_energy,
        'xs': exp3_xs,
        'dxs': exp3_err,
        'facility': 'GELINA'
    })
    
    return experiments

# Create sample EXFOR data
exfor_data = create_exfor_sample_data()

print("Sample EXFOR data created:")
print(f"Number of experiments: {len(exfor_data)}")
for exp in exfor_data:
    print(f"\n  Entry: {exp['entry']}")
    print(f"  Author: {exp['author']} ({exp['year']})")
    print(f"  Facility: {exp['facility']}")
    print(f"  Data points: {len(exp['energy'])}")

## 5. Visualizing Experimental Data with Uncertainties

A key feature of experimental data is the uncertainty bars, which represent measurement errors.

In [ ]:
plt.figure(figsize=(14, 8))

colors = ['red', 'blue', 'green']
markers = ['o', 's', '^']

for i, exp in enumerate(exfor_data):
    plt.errorbar(
        exp['energy'], 
        exp['xs'], 
        yerr=exp['dxs'],
        fmt=markers[i],
        color=colors[i],
        label=f"{exp['author']} ({exp['year']}) - {exp['entry']}",
        markersize=6,
        capsize=3,
        alpha=0.7
    )

plt.xscale('log')
plt.yscale('log')
plt.xlabel('Neutron Energy (eV)', fontsize=14)
plt.ylabel('Fission Cross-section (barns)', fontsize=14)
plt.title('U-235 Fission Cross-Section: Experimental Data from EXFOR', fontsize=16)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.savefig(data_dir / 'exfor_u235_fission.png', dpi=150)
plt.show()

print("Plot saved to:", data_dir / 'exfor_u235_fission.png')

## 6. Comparing EXFOR with ENDF Evaluation

Let's overlay experimental EXFOR data with the evaluated ENDF curve to see how well they agree.

In [ ]:
# Create a smooth ENDF-like evaluation curve
energy_eval = np.logspace(-2, 7, 10000)

# Simulated ENDF evaluation (in practice, use actual ENDF data from Tutorial 1)
def endf_u235_fission(E):
    """Simplified model of U-235 fission cross-section"""
    # Thermal region: 1/v behavior
    thermal = 584.4 * np.sqrt(0.0253 / np.maximum(E, 1e-5))
    
    # Resonance region: simplified resonance structure
    resonance = 50 + 30 * np.sin(np.log(np.maximum(E, 1)))
    
    # Fast region: slowly varying
    fast = 2.0 + 0.5 * np.log10(np.maximum(E, 1e5) / 1e5)
    
    # Smooth transition between regions
    xs = np.where(E < 1, thermal, np.where(E < 1e4, resonance, fast))
    return xs

xs_eval = endf_u235_fission(energy_eval)

# Plot
plt.figure(figsize=(14, 8))

# Plot ENDF evaluation as a line
plt.plot(energy_eval, xs_eval, 'k-', linewidth=2, label='ENDF Evaluation', alpha=0.6)

# Plot experimental data points
for i, exp in enumerate(exfor_data):
    plt.errorbar(
        exp['energy'], 
        exp['xs'], 
        yerr=exp['dxs'],
        fmt=markers[i],
        color=colors[i],
        label=f"EXFOR: {exp['author']} ({exp['year']})",
        markersize=6,
        capsize=3,
        alpha=0.7
    )

plt.xscale('log')
plt.yscale('log')
plt.xlabel('Neutron Energy (eV)', fontsize=14)
plt.ylabel('Fission Cross-section (barns)', fontsize=14)
plt.title('U-235 Fission: ENDF Evaluation vs EXFOR Experimental Data', fontsize=16)
plt.legend(fontsize=10, loc='best')
plt.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.savefig(data_dir / 'exfor_vs_endf.png', dpi=150)
plt.show()

## 7. Statistical Analysis of Experimental Data

Let's analyze the scatter and uncertainties in the experimental data.

In [ ]:
# Combine all experimental data into a DataFrame
all_data = []
for exp in exfor_data:
    for j in range(len(exp['energy'])):
        all_data.append({
            'Entry': exp['entry'],
            'Author': exp['author'],
            'Year': exp['year'],
            'Energy_eV': exp['energy'][j],
            'CrossSection_barns': exp['xs'][j],
            'Uncertainty_barns': exp['dxs'][j],
            'Relative_Uncertainty_%': (exp['dxs'][j] / exp['xs'][j]) * 100
        })

df_exfor = pd.DataFrame(all_data)

print("EXFOR Data Summary:")
print(df_exfor.describe())

# Save to CSV
csv_file = data_dir / 'exfor_u235_fission.csv'
df_exfor.to_csv(csv_file, index=False)
print(f"\nData saved to: {csv_file}")

## 8. Uncertainty Visualization

Let's visualize how uncertainties vary across different energy ranges.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Plot 1: Absolute uncertainties
for i, exp in enumerate(exfor_data):
    ax1.loglog(
        exp['energy'], 
        exp['dxs'],
        markers[i],
        color=colors[i],
        label=f"{exp['author']} ({exp['year']})",
        markersize=8
    )

ax1.set_xlabel('Neutron Energy (eV)', fontsize=12)
ax1.set_ylabel('Absolute Uncertainty (barns)', fontsize=12)
ax1.set_title('Absolute Uncertainties in Experimental Data', fontsize=14)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3, which='both')

# Plot 2: Relative uncertainties
for i, exp in enumerate(exfor_data):
    rel_unc = (exp['dxs'] / exp['xs']) * 100
    ax2.semilogx(
        exp['energy'], 
        rel_unc,
        markers[i],
        color=colors[i],
        label=f"{exp['author']} ({exp['year']})",
        markersize=8
    )

ax2.set_xlabel('Neutron Energy (eV)', fontsize=12)
ax2.set_ylabel('Relative Uncertainty (%)', fontsize=12)
ax2.set_title('Relative Uncertainties in Experimental Data', fontsize=14)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(data_dir / 'exfor_uncertainties.png', dpi=150)
plt.show()

## 9. Real EXFOR Data Access

### Using the IAEA EXFOR Retrieval System

For accessing real EXFOR data, you have several options:

#### Option 1: Web Interface
Visit: https://www-nds.iaea.org/exfor/
- Search by reaction, isotope, or author
- Download data in various formats (C4, X4, CSV)

#### Option 2: Python Package x4i3
```python
# Install: pip install x4i3
import x4i3

# Search for U-235 fission data
results = x4i3.search(target='U-235', reaction='(n,f)')
```

#### Option 3: JANIS Interface (See Tutorial 4)
The JANIS system provides a unified interface to multiple databases including EXFOR.

### Example: Manual Download
1. Go to https://www-nds.iaea.org/exfor/
2. Search for: Target="92-U-235", Reaction="(N,F)"
3. Download data in CSV format
4. Load into pandas for analysis

## 10. Machine Learning Applications

EXFOR data is valuable for machine learning because:

1. **Training Data**: Use experimental measurements to train ML models
2. **Validation**: Compare ML predictions with independent measurements
3. **Outlier Detection**: Identify potentially problematic measurements
4. **Uncertainty Quantification**: Learn patterns in experimental uncertainties
5. **Data Consistency**: Check consistency between different experiments

In [ ]:
# Example: Simple outlier detection
from scipy import stats

# Calculate residuals between experiments and evaluation
residuals = []
for exp in exfor_data:
    xs_eval_at_exp = endf_u235_fission(exp['energy'])
    residual = (exp['xs'] - xs_eval_at_exp) / exp['dxs']  # Normalized residual
    residuals.extend(residual)

residuals = np.array(residuals)

plt.figure(figsize=(12, 6))
plt.hist(residuals, bins=20, edgecolor='black', alpha=0.7)
plt.axvline(0, color='red', linestyle='--', linewidth=2, label='Perfect agreement')
plt.axvline(np.mean(residuals), color='blue', linestyle='--', linewidth=2, 
            label=f'Mean = {np.mean(residuals):.2f}')
plt.xlabel('Normalized Residual (σ)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.title('Distribution of Residuals: EXFOR vs ENDF', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Mean residual: {np.mean(residuals):.2f} σ")
print(f"Std deviation: {np.std(residuals):.2f} σ")
print(f"\nPoints beyond 3σ: {np.sum(np.abs(residuals) > 3)}")

## 11. Key Takeaways

1. **EXFOR contains experimental data** - Raw measurements with uncertainties
2. **Multiple experiments exist** for the same reaction - Provides validation
3. **Uncertainties are crucial** - They quantify measurement quality
4. **Scatter between experiments** is normal - Due to different methods, facilities, etc.
5. **EXFOR is the foundation** for evaluated libraries like ENDF

## Next Steps

- Download real EXFOR data for your isotope of interest
- Compare multiple evaluations (ENDF, JEFF, JENDL) with EXFOR
- Use EXFOR data for ML model training and validation
- Investigate discrepancies between experiments

## Resources

- EXFOR Database: https://www-nds.iaea.org/exfor/
- x4i3 Python Package: https://github.com/afedynitch/x4i3
- IAEA Nuclear Data Services: https://www-nds.iaea.org/
- EXFOR Manual: https://www-nds.iaea.org/nrdc/exfor-man/